# 06 · Subqueries

A subquery is a query nested inside another. Flavors:
- **scalar** subquery → returns one value
- subquery with `IN`
- **correlated** subquery → references the outer query
- `EXISTS`
- subquery in `FROM` (a *derived table*)

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Scalar subquery
Products priced above the overall average price. The inner query returns a single number.

In [ ]:
%%sql
SELECT product_name, unit_price
FROM products
WHERE unit_price > (SELECT AVG(unit_price) FROM products)
ORDER BY unit_price DESC;

## Subquery with `IN`
Customers who have placed at least one order (their id appears in `orders`).

In [ ]:
%%sql
SELECT first_name, last_name
FROM customers
WHERE customer_id IN (SELECT customer_id FROM orders);

## `NOT IN` — the opposite
Customers who have never ordered:

In [ ]:
%%sql
SELECT first_name, last_name
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders);

## Correlated subquery
The inner query runs *per outer row* and references it. Count each customer's
orders inline:

In [ ]:
%%sql
SELECT cu.first_name, cu.last_name,
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = cu.customer_id) AS order_count
FROM customers AS cu
ORDER BY order_count DESC;

## `EXISTS`
Often clearer/faster than `IN` for "does a related row exist?". Products that
have actually been sold:

In [ ]:
%%sql
SELECT p.product_name
FROM products AS p
WHERE EXISTS (SELECT 1 FROM order_items oi WHERE oi.product_id = p.product_id)
ORDER BY p.product_name;

## Subquery in `FROM` (derived table)
Compute per-order totals first, then filter on them. You must alias the derived
table.

In [ ]:
%%sql
SELECT order_id, order_total
FROM (
    SELECT order_id, SUM(quantity * unit_price) AS order_total
    FROM order_items
    GROUP BY order_id
) AS totals
WHERE order_total > 200
ORDER BY order_total DESC;

## Practice

**✏️ Exercise 1.** Find products that are cheaper than the average price of their own... keep it simple: products cheaper than the overall average price.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT product_name, unit_price
FROM products
WHERE unit_price < (SELECT AVG(unit_price) FROM products)
ORDER BY unit_price;

**✏️ Exercise 2.** List employees who have handled at least one order (use IN or EXISTS against the orders table).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, last_name
FROM employees e
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.employee_id = e.employee_id);

**✏️ Exercise 3.** Using a derived table, find the average order total across all orders.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT order_id, SUM(quantity * unit_price) AS order_total
    FROM order_items
    GROUP BY order_id
) AS t;

### ✅ Recap
Subqueries let one query feed another: scalars for comparisons, `IN`/`EXISTS`
for membership, correlated subqueries for per-row logic, and derived tables to
query an intermediate result.

**Next:** `07_set_operations.ipynb`.